# Ingestão e limpeza - Diesel (ANP)

Pipeline de ingestão do preço de diesel (série histórica ANP), usado como proxy de custo de frete no projeto. Fonte e racional documentados em `data/raw/diesel/base.yaml`.

```text
data/raw/diesel/base.yaml (config)
    -> download_all()        data/raw/diesel/data/*.csv
    -> convert_to_parquet()  data/raw/diesel/data/diesel.parquet
    -> clean()                data/interim/diesel/diesel.parquet
    -> download_ipca()          data/raw/ipca/ipca.csv (número-índice IPCA, API SIDRA)
    -> deflate()                 data/interim/diesel/diesel_deflacionado.parquet
                                  (adiciona valor_de_venda_real; diesel.parquet nominal fica intacto)
    -> aggregate()                 data/processed/diesel/*.parquet
    -> push_diesel()                  Hugging Face Hub (pbf-feijao-mecai-usp/bf-feijao-dados)
```

As células de `download_all`/`convert_to_parquet` estão comentadas por padrão — são operações pesadas (~2,5GB / 31 arquivos) e só precisam rodar de novo se os dados brutos não existirem ainda ou para atualizar a série. `clean`/`download_ipca`/`deflate`/`aggregate` são baratas e idempotentes, podem rodar sempre. `push_diesel` também está comentada por padrão — sobe os dados pro dataset público compartilhado do time, então só deve rodar quando você quiser mesmo publicar uma atualização (precisa de `huggingface_token` no `.env`, ver README seção 15.1).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "pyproject.toml").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.diesel import aggregate, clean, convert_to_parquet, deflate, download_all, download_ipca
from src.hub import push_diesel

In [2]:
# Pesado (~2,5GB, 31 arquivos) - rodar só se data/raw/diesel/data/ ainda não existir
# ou para atualizar a série com arquivos novos publicados pela ANP.
# download_all()
# convert_to_parquet()

In [3]:
clean()
download_ipca()
deflate()
aggregate()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total: 8,196,141 linhas
Parquet salvo em /Users/alessandrooliveira/Estudos/MECAI/mai5003/mecai-26-prob-estat/data/interim/diesel/diesel.parquet


IPCA salvo em /Users/alessandrooliveira/Estudos/MECAI/mai5003/mecai-26-prob-estat/data/raw/ipca/ipca.csv


Total: 8,196,141 linhas
Parquet salvo em /Users/alessandrooliveira/Estudos/MECAI/mai5003/mecai-26-prob-estat/data/interim/diesel/diesel_deflacionado.parquet
mensal_geral: 264 linhas -> /Users/alessandrooliveira/Estudos/MECAI/mai5003/mecai-26-prob-estat/data/processed/diesel/mensal_geral.parquet


mensal_estado: 7,127 linhas -> /Users/alessandrooliveira/Estudos/MECAI/mai5003/mecai-26-prob-estat/data/processed/diesel/mensal_estado.parquet
mensal_regiao: 1,320 linhas -> /Users/alessandrooliveira/Estudos/MECAI/mai5003/mecai-26-prob-estat/data/processed/diesel/mensal_regiao.parquet
variacao_anual: 23 linhas -> /Users/alessandrooliveira/Estudos/MECAI/mai5003/mecai-26-prob-estat/data/processed/diesel/variacao_anual.parquet


In [4]:
# Sobe os dados gerados acima pro dataset público do time no Hugging Face Hub.
# Precisa de huggingface_token (escrita) no .env - ver README seção 15.1.
# push_diesel()